# 02 — Train ViT & ResNet (change MODEL and FRACTION each run)

**Runtime → T4 GPU** recommended for ViT.

If you already ran notebook **01** (setup + `casting_data.zip` upload), skip to **Cell 4**. Otherwise run **Cells 2–3** first (same as `COLAB_01_EVERY_CELL.md`).


In [1]:
%cd /content

!pip install -q pillow pandas matplotlib scikit-learn pyyaml torch torchvision transformers

!rm -rf /content/casting-defect-vit-resnet
!git clone https://github.com/anshikareddy403/casting-defect-vit-resnet.git

%cd /content/casting-defect-vit-resnet
!ls src/

import os
import sys
from pathlib import Path

ROOT = Path('/content/casting-defect-vit-resnet')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

print('Setup OK — data.py should appear in the ls output above')


/content
Cloning into 'casting-defect-vit-resnet'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 64 (delta 11), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (64/64), 485.38 KiB | 5.85 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/casting-defect-vit-resnet
data.py      inference.py  metrics.py	train_resnet.py  utils.py
evaluate.py  __init__.py   __pycache__	train_vit.py	 visualize.py
Setup OK — data.py should appear in the ls output above


In [2]:
from google.colab import files
from pathlib import Path

raw = ROOT / 'data' / 'raw'
raw.mkdir(parents=True, exist_ok=True)

print('Click Choose Files and select casting_data.zip from your computer')
uploaded = files.upload()

for name, data in uploaded.items():
    dest = raw / name
    dest.write_bytes(data)
    print('Saved', dest, 'bytes', dest.stat().st_size)

zip_path = raw / 'casting_data.zip'
assert zip_path.exists(), 'Upload casting_data.zip first'

!unzip -q -o /content/casting-defect-vit-resnet/data/raw/casting_data.zip -d /content/casting-defect-vit-resnet/data/raw
!ls /content/casting-defect-vit-resnet/data/raw/casting_data/train/


Click Choose Files and select casting_data.zip from your computer


Saving casting_data.zip to casting_data.zip
Saved /content/casting-defect-vit-resnet/data/raw/casting_data.zip bytes 72479122
def_front  ok_front


In [4]:
import os
import sys
from pathlib import Path

import pandas as pd
import torch

ROOT = Path('/content/casting-defect-vit-resnet')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src.data import (
    load_all_samples,
    make_loaders,
    resolve_data_root,
    split_train_val,
    subset_train_stratified,
)
from src.evaluate import evaluate_model
from src.metrics import format_metrics_row
from src.train_resnet import train_resnet
from src.train_vit import train_vit
from src.utils import ensure_dir, load_config, resolve_device, set_seed
from src.visualize import plot_confusion_matrix, plot_training_history, save_gradcam_examples

# === CHANGE THESE FOR EACH RUN ===
FRACTION = 1.0      # 1.0, then 0.5, then 0.25
MODEL = 'resnet'    # 'resnet' or 'vit'
# =================================

cfg = load_config(ROOT / 'configs' / 'default.yaml')
# Disable group augmentations to prevent stratification errors on small datasets
cfg['data']['group_augmentations'] = False

set_seed(cfg['seed'])
device = resolve_device(cfg.get('device', 'auto'))
print('Device:', device, '| MODEL:', MODEL, '| FRACTION:', FRACTION)

data_root = resolve_data_root(cfg)
train_pool, test_samples, layout = load_all_samples(data_root, layout=cfg['data'].get('layout', 'auto'))
train_samples, val_samples = split_train_val(
    train_pool,
    val_fraction=cfg['data']['val_fraction'],
    seed=cfg['seed'],
    group_augmentations=cfg['data'].get('group_augmentations', False),
)
train_subset = subset_train_stratified(
    train_samples,
    fraction=FRACTION,
    seed=cfg['seed'],
    group_augmentations=cfg['data'].get('group_augmentations', False),
)
train_loader, val_loader, test_loader = make_loaders(
    train_subset,
    val_samples,
    test_samples,
    image_size=cfg['data']['image_size'],
    batch_size=cfg['data']['batch_size'],
)
print(f'n_train={len(train_subset)} layout={layout}')

Device: cuda | MODEL: resnet | FRACTION: 1.0
n_train=5638 layout=kaggle_split


In [6]:
run_name = f"{MODEL}_frac{FRACTION}_seed{cfg['seed']}"
ckpt_path = ensure_dir(ROOT / cfg['paths']['checkpoints'] / run_name) / 'best.pt'

if MODEL == 'resnet':
    model, train_meta, history = train_resnet(train_loader, val_loader, device, cfg, ckpt_path)
    is_vit = False
else:
    model, train_meta, history = train_vit(train_loader, val_loader, device, cfg, ckpt_path)
    is_vit = True

eval_out = evaluate_model(model, test_loader, device, is_vit=is_vit)
row = format_metrics_row(
    model=MODEL,
    train_fraction=FRACTION,
    metrics=eval_out['metrics'],
    train_time_s=train_meta['train_time_s'],
    infer_latency_ms=eval_out['infer_latency_ms'],
    model_size_mb=eval_out['model_size_mb'],
    seed=cfg['seed'],
)
display(pd.DataFrame([row]))

csv_path = ROOT / cfg['paths']['comparison_csv']
csv_path.parent.mkdir(parents=True, exist_ok=True)

try:
    from src.metrics import append_metrics_csv
    csv_path = append_metrics_csv(
        row,
        csv_path,
        dataset_layout=layout,
        n_train=len(train_subset),
        n_val=len(val_samples),
        n_test=len(test_samples),
    )
except ImportError:
    # Safe fallback if append_metrics_csv is missing from src.metrics
    import csv

    # Enrich the row with split statistics
    row_copy = row.copy()
    row_copy['dataset_layout'] = layout
    row_copy['n_train'] = len(train_subset)
    row_copy['n_val'] = len(val_samples)
    row_copy['n_test'] = len(test_samples)

    file_exists = csv_path.exists()
    fieldnames = list(row_copy.keys())

    with open(csv_path, mode='a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerow(row_copy)

print('Appended to', csv_path)

test: 100%|██████████| 23/23 [00:02<00:00,  9.83it/s]


,model,train_fraction,seed,accuracy,macro_f1,defect_precision,defect_recall,tn,fp,fn,tp,train_time_s,infer_latency_ms_mean,model_size_mb
0,resnet,1.0,42,0.997203,0.996993,1.0,0.995585,262,0,2,451,212.931888,0.939151,42.675697


Appended to /content/casting-defect-vit-resnet/results/comparison_table.csv


In [12]:
import torch
import src.visualize

# Dynamic monkeypatch to fix the 5D tensor dimension bug in ResNetGradCAM
original_call = src.visualize.ResNetGradCAM.__call__

def patched_call(self, image_tensor, class_idx=None):
    self.model.eval()
    # Only unsqueeze if the image is 3D (unbatched)
    if image_tensor.dim() == 3:
        image_tensor = image_tensor.unsqueeze(0)

    logits = self.model(image_tensor)
    if class_idx is None:
        class_idx = int(logits.argmax(dim=1).item())

    # Use the rest of the original logic
    loss = logits[0, class_idx]
    self.model.zero_grad()
    loss.backward()

    grads = self.gradients.cpu().data.numpy()[0]
    features = self.activations.cpu().data.numpy()[0]

    weights = torch.nn.functional.adaptive_avg_pool2d(self.gradients, 1).cpu().data.numpy()[0]
    cam = torch.zeros(features.shape[1:], dtype=torch.float32).numpy()

    for i, w in enumerate(weights):
        cam += w * features[i]

    import numpy as np
    cam = np.maximum(cam, 0)
    if cam.max() > 0:
        cam = cam / cam.max()
    return cam

src.visualize.ResNetGradCAM.__call__ = patched_call

fig_dir = ensure_dir(ROOT / cfg['paths']['figures'] / run_name)
plot_training_history(history, fig_dir / 'training_curves.png', title=run_name)
plot_confusion_matrix(eval_out['metrics']['confusion_matrix'], fig_dir / 'confusion_matrix.png', title=run_name)

# Pass arguments in correct order with a specified .png filename
save_gradcam_examples(model, test_loader, fig_dir / 'gradcam.png', device)

print('Figures saved under', fig_dir)

Figures saved under /content/casting-defect-vit-resnet/results/figures/resnet_frac1.0_seed42


## Six runs (data-efficiency grid)

1. Edit **MODEL** and **FRACTION** in Cell 4.
2. Run **Cell 4 → Cell 5 → Cell 6** (Cell 6 plots are optional).
3. Repeat for all six combinations:

| MODEL | FRACTION |
|-------|----------|
| resnet | 1.0, 0.5, 0.25 |
| vit | 1.0, 0.5, 0.25 |

Download `results/comparison_table.csv` from the Files pane (or push to GitHub) for the report and Streamlit **Results** tab.

**Optional — full grid via CLI** (from repo root on a machine with data already in place):

```bash
for m in resnet vit; do for f in 1.0 0.5 0.25; do python scripts/run_experiment.py --model $m --fraction $f; done; done
```


In [15]:
import subprocess
import yaml
from pathlib import Path

# Safely disable group_augmentations in default.yaml config to prevent split errors
cfg_path = Path('/content/casting-defect-vit-resnet/configs/default.yaml')
if cfg_path.exists():
    with open(cfg_path, 'r') as f:
        cfg_data = yaml.safe_load(f)

    cfg_data['data']['group_augmentations'] = False

    with open(cfg_path, 'w') as f:
        yaml.safe_dump(cfg_data, f)
    print("Disabled group_augmentations in config file.")

# Define the grid parameters
models = ['resnet', 'vit']
fractions = [1.0, 0.5, 0.25]

# Run each experiment via the CLI script
for model in models:
    for fraction in fractions:
        print(f"=== Running Experiment: MODEL={model}, FRACTION={fraction} ===")
        cmd = f"python scripts/run_experiment.py --model {model} --fraction {fraction}"
        process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

        # Stream the output live to the notebook console
        for line in process.stdout:
            print(line, end='')
        process.wait()
        print(f"=== Finished: {model} at {fraction} ===\n")

Streaming output truncated to the last 5000 lines.
train:  98%|█████████▊| 44/45 [00:05<00:00,  8.82it/s]
                                                      

eval:  97%|█████████▋| 31/32 [00:02<00:00, 12.13it/s]
                                                     

train:  98%|█████████▊| 44/45 [00:05<00:00,  8.74it/s]
                                                      

eval:  97%|█████████▋| 31/32 [00:02<00:00, 13.21it/s]
                                                     

train:  98%|█████████▊| 44/45 [00:05<00:00,  8.68it/s]
                                                      

eval:  97%|█████████▋| 31/32 [00:02<00:00, 11.90it/s]
                                                     

train:  98%|█████████▊| 44/45 [00:05<00:00,  8.27it/s]
                                                      

eval:  97%|█████████▋| 31/32 [00:02<00:00, 13.69it/s]
                                                     

train:  98%|█████████▊| 44/45 [00:05<00:00,  8.78it/s]
              

In [16]:
import pandas as pd
# Display the final aggregated comparison table
csv_path = '/content/casting-defect-vit-resnet/results/comparison_table.csv'
if pd.io.common.file_exists(csv_path):
    df = pd.read_csv(csv_path)
    display(df)
else:
    print("Comparison table not found. Verify if experiments ran successfully.")

,model,train_fraction,seed,accuracy,macro_f1,defect_precision,defect_recall,tn,fp,fn,tp,train_time_s,infer_latency_ms_mean,model_size_mb,dataset_layout,n_train,n_val,n_test
0,resnet,1.00,42,0.997203,0.996993,1.000000,0.995585,262,0,2,451,212.931888,0.939151,42.675697,kaggle_split,5638,995,715
1,resnet,1.00,42,0.997203,0.996993,1.000000,0.995585,262,0,2,451,305.876625,0.921554,42.675697,kaggle_split,5638,995,715
2,resnet,0.50,42,0.995804,0.995493,1.000000,0.993377,262,0,3,450,108.184477,0.939314,42.675697,kaggle_split,2819,995,715
3,resnet,0.25,42,0.997203,0.996993,1.000000,0.995585,262,0,2,451,118.258879,0.925171,42.675697,kaggle_split,1409,995,715
4,vit,1.00,42,0.997203,0.996988,0.997792,0.997792,261,1,1,452,1780.973167,12.304858,327.301765,kaggle_split,5638,995,715
5,vit,0.50,42,0.994406,0.993976,0.995585,0.995585,260,2,2,451,1533.803993,12.271673,327.301765,kaggle_split,2819,995,715
6,vit,0.25,42,0.994406,0.993985,0.997783,0.993377,261,1,3,450,533.875301,12.405679,327.301765,kaggle_split,1409,995,715
